# Week 4 - Learning rules - Exercises

This is an outline of the exercise only, leaving you a lot of details to fill in on your own. Later I will provide a more detailed version with some hints.

## Task 1 - Implement STDP

Implement a simulator including STDP using the efficient method from the STDP video.

## Task 2 - Competition and latency

Use your simulator to reproduce the competition and latency reduction results from this paper:

[Song et al. (2000) "Competitive Hebbian learning through spike-timing-dependent synaptic plasticity"](https://doi.org/10.1038/78829)

## Task 3 - Correlations

Use your simulator to reproduce the correlated groups results from this paper:

[Song and Abbott (2001) "Cortical Development and Remapping through Spike Timing-Dependent Plasticity"](https://doi.org/10.1016/S0896-6273(01)00451-2)

## Task 4 - Sequences (optional)

Use your simulator to reproduce the sequence learning results from this paper:

[Masquelier et al. (2008) "Spike Timing Dependent Plasticity Finds the Start of Repeating Patterns in Continuous Spike Trains"](https://doi.org/10.1371/journal.pone.0001377)

This might be quite challenging because this paper uses a different version of the STDP learning rule. For more details on how to reproduce this result, see:

[Hathway and Goodman (2018) "[Re] Spike Timing Dependent Plasticity Finds the Start of Repeating Patterns in Continuous Spike Trains"](http://neural-reckoning.org/pub_re_stdp_repeating_patterns.html)

Can you extend the results of these two papers to investigate how robust the results are to different versions of STDP, different parameters, etc.?

In [1]:
import numpy as np

In [7]:
# pre & post synaptic spikes generation
duration = 1000  # ms
dt = 10 # ms
delay = 10 #ms
num_steps = int(duration/dt)
num_neu = 2
diff = int(delay/dt)
spikes = np.zeros((num_neu, num_steps))
num_spikes = int(num_steps/10)
spikes[0, :num_spikes-1] = 1
np.random.shuffle(spikes[0, :])
for i in range (num_steps):
  if i+diff >= num_steps:
    break
  spikes[1, i+diff] = spikes[0, i]

print(spikes)

[[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.
  0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.
  0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.
  0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  1. 0. 0. 0.]]


In [8]:
# weight update
tau_plus = 20
tau_minus = 20
a_plus = 0.01
a_minus = -0.012
a_pre = 0
a_post = 0
w = 0.1
w_history = [w]

for i in range(num_steps):
    a_pre *= np.exp(-dt / tau_plus)
    a_post *= np.exp(-dt / tau_minus)
    pre_spike = spikes[0, i] == 1
    post_spike = spikes[1, i] == 1
    if pre_spike:
        w += a_post
    if post_spike:
        w += a_pre
    if pre_spike:
        a_pre += a_plus
    if post_spike:
        a_post += a_minus
    w_history.append(w)
    print(w)

print("Initial weight:", w_history[0])
print("Final weight:", w)

0.1
0.1
0.1
0.1
0.1
0.1
0.1
0.1
0.1
0.10606530659712635
0.10606530659712635
0.10606530659712635
0.10338774467534519
0.1102739012587105
0.1102739012587105
0.1102739012587105
0.1102739012587105
0.10843009019320635
0.11506064694400217
0.11506064694400217
0.11506064694400217
0.11506064694400217
0.11506064694400217
0.11506064694400217
0.11506064694400217
0.11466450776527952
0.12085125724548469
0.12085125724548469
0.12085125724548469
0.12085125724548469
0.12085125724548469
0.12085125724548469
0.12085125724548469
0.12085125724548469
0.12085125724548469
0.12085125724548469
0.12085125724548469
0.12085125724548469
0.12085125724548469
0.12085125724548469
0.12085125724548469
0.12085125724548469
0.12085125724548469
0.12085125724548469
0.12085125724548469
0.12085125724548469
0.12085125724548469
0.12085125724548469
0.12085125724548469
0.12085125724548469
0.12085125724548469
0.12085125724548469
0.12085125724548469
0.12085125724548469
0.12085124706734819
0.12691655678474717
0.12691655678474717
0.126916

In [22]:
# LIF Neuron
def LIF(I, tau=10, threshold=1.0, reset=0.0, dt=dt):
    I = np.asarray(I, dtype=float)
    num_steps = len(I)
    V = 0.0
    V_rec = np.zeros(num_steps)
    decay = np.exp(-dt / tau)
    spikes = [] # list to store spike times
    for k in range(num_steps):
        # Record V(k * dt)
        V_rec[k] = V

        # Exact solution over one time step, assuming I[k] is constant:
        # V(t + dt) = I[k] + (V(t) - I[k]) * exp(-dt / tau)
        V = I[k] + (V - I[k]) * decay

        # Check whether the updated voltage crosses the threshold
        if V >= threshold:
            # The spike occurs at the end of the current time step
            spikes.append((k + 1) * dt)

            # Reset the membrane potential
            V = reset
    return V_rec, np.array(spikes)

In [30]:
def cur(w, spike, I0):
  I_k = np.zeros(len(spike))
  for i in range (len(spike)):
    if spike[i] == 1:
      I_k[i] = w
  I = I_k + I0
  return I
T = np.arange(num_steps)*dt*1e-3
I = cur(w, spikes[0, :], 3*np.sin(2*np.pi*10*T)**2)

V_neu1, S_neu1 = LIF(I, 10)
print(V_neu1)
print(S_neu1)

[0.         0.         0.65517685 0.         0.         0.65517685
 0.24102609 0.74384539 0.         0.         0.65517685 0.24102609
 0.74384539 0.         0.         0.65517685 0.24102609 0.74384539
 0.         0.         0.65517685 0.24102609 0.74384539 0.
 0.         0.65517685 0.33098703 0.77694017 0.         0.
 0.65517685 0.24102609 0.74384539 0.         0.         0.65517685
 0.24102609 0.74384539 0.         0.         0.65517685 0.24102609
 0.74384539 0.         0.         0.65517685 0.24102609 0.74384539
 0.         0.         0.65517685 0.24102609 0.74384539 0.
 0.         0.74513779 0.27412087 0.75602028 0.         0.
 0.65517685 0.24102609 0.74384539 0.         0.         0.74513779
 0.27412087 0.75602028 0.         0.         0.65517685 0.24102609
 0.74384539 0.         0.         0.65517685 0.24102609 0.74384539
 0.         0.         0.65517685 0.24102609 0.74384539 0.
 0.         0.65517685 0.24102609 0.74384539 0.         0.
 0.65517685 0.24102609 0.74384539 0.       